# Direct CT Prediction Pipeline

Predict segment-level CT for the next 14 days using historical CT, predicted MV, and day type features.

- **Model**: `direct_ct` — LSTM with shared backbone + per-segment heads
- **Baseline**: `baseline_method` — 5-3-1-1 weighted average
- **Evaluation**: MAPE on test period (2026-01-15 to 2026-03-01)

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Constants
SEGMENTS = [f'G{i}' for i in range(1, 15)]
N_SEGMENTS = len(SEGMENTS)
LOOKBACK = 28
HORIZON = 14
DATE_START = '2025-01-01'
DATE_END = '2026-03-01'
TEST_START = '2026-01-15'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Dummy Data Generation

In [ ]:
dates = pd.date_range(DATE_START, DATE_END, freq='D')
n_days = len(dates)

# Per-segment parameters
np.random.seed(SEED)
base_mv = np.random.uniform(500, 2000, N_SEGMENTS)
base_ct = np.random.uniform(3600, 36000, N_SEGMENTS)
alpha = np.random.uniform(-2.0, 5.0, N_SEGMENTS)  # CT-MV correlation strength

# US federal holidays (simplified set)
holidays_list = pd.to_datetime([
    # 2025
    '2025-01-01', '2025-01-20', '2025-02-17', '2025-05-26',
    '2025-07-04', '2025-09-01', '2025-10-13', '2025-11-11',
    '2025-11-27', '2025-12-25',
    # 2026
    '2026-01-01', '2026-01-19', '2026-02-16',
])

# Day type: 0=workday, 1=holiday, 2=day_after_holiday
day_type_arr = np.zeros(n_days, dtype=int)
for i, d in enumerate(dates):
    if d in holidays_list or d.dayofweek >= 5:  # weekends + holidays
        day_type_arr[i] = 1
for i in range(1, n_days):
    if day_type_arr[i] == 0 and day_type_arr[i-1] == 1:
        day_type_arr[i] = 2

day_type_df = pd.DataFrame({'hist_date': dates, 'day_type': day_type_arr})

# Generate MV
mv_data = {}
for s, seg in enumerate(SEGMENTS):
    t = np.arange(n_days)
    weekly = 80 * np.sin(2 * np.pi * t / 7)  # weekly seasonality
    trend = 0.3 * t  # slight upward trend
    noise = np.random.normal(0, 50, n_days)
    # Holiday effect: lower MV on holidays
    holiday_effect = np.where(day_type_arr == 1, -base_mv[s] * 0.3, 0)
    holiday_effect += np.where(day_type_arr == 2, -base_mv[s] * 0.1, 0)
    mv_data[seg] = np.maximum(base_mv[s] + weekly + trend + noise + holiday_effect, 50)

mv_pred = pd.DataFrame({'hist_date': dates, **mv_data})

# Generate CT (correlated with MV)
ct_data = {}
for s, seg in enumerate(SEGMENTS):
    weekly = 500 * np.sin(2 * np.pi * np.arange(n_days) / 7 + np.random.uniform(0, np.pi))
    # Random walk drift
    drift = np.cumsum(np.random.normal(0, 10, n_days))
    noise = np.random.normal(0, 200, n_days)
    # Holiday effect: higher CT on holidays/day-after
    holiday_ct = np.where(day_type_arr == 1, base_ct[s] * 0.15, 0)
    holiday_ct += np.where(day_type_arr == 2, base_ct[s] * 0.08, 0)
    ct_data[seg] = np.maximum(
        base_ct[s] + alpha[s] * mv_data[seg] + weekly + drift + noise + holiday_ct,
        500
    )

ct_gt = pd.DataFrame({'hist_date': dates, **ct_data})

print(f'ct_gt shape: {ct_gt.shape}')
print(f'mv_pred shape: {mv_pred.shape}')
print(f'Date range: {dates[0].date()} to {dates[-1].date()} ({n_days} days)')
print(f'Day type distribution: workday={np.sum(day_type_arr==0)}, holiday={np.sum(day_type_arr==1)}, day_after={np.sum(day_type_arr==2)}')
ct_gt.head()

## 3. Data Exploration

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
plot_segs = SEGMENTS[:4]

for i, seg in enumerate(plot_segs):
    # CT
    axes[i, 0].plot(ct_gt['hist_date'], ct_gt[seg], linewidth=0.7)
    axes[i, 0].axvline(pd.Timestamp(TEST_START), color='r', linestyle='--', alpha=0.7, label='Test start')
    axes[i, 0].set_title(f'{seg} - CT (seconds)')
    axes[i, 0].legend()
    # MV
    axes[i, 1].plot(mv_pred['hist_date'], mv_pred[seg], linewidth=0.7, color='orange')
    axes[i, 1].axvline(pd.Timestamp(TEST_START), color='r', linestyle='--', alpha=0.7, label='Test start')
    axes[i, 1].set_title(f'{seg} - MV')
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

# Correlation
print('\nCT-MV Pearson correlation per segment:')
for seg in SEGMENTS:
    corr = ct_gt[seg].corr(mv_pred[seg])
    print(f'  {seg}: {corr:.3f}')

## 4. Baseline: `baseline_method` (5-3-1-1 Weighted Average)

In [ ]:
def baseline_method(ct_history):
    """5-3-1-1 weighted average. Input: last 28 days of CT. Output: single predicted value repeated for 14 days."""
    w1 = ct_history[-7:].mean()    # most recent week
    w2 = ct_history[-14:-7].mean() # 2nd week back
    w3 = ct_history[-21:-14].mean() # 3rd week back
    w4 = ct_history[-28:-21].mean() # 4th week back
    pred = (5 * w1 + 3 * w2 + 1 * w3 + 1 * w4) / 10.0
    return np.full(HORIZON, pred)

# Generate baseline predictions for all test origins
ct_arr = ct_gt[SEGMENTS].values  # (n_days, 14)
date_arr = ct_gt['hist_date'].values
test_mask = ct_gt['hist_date'] >= TEST_START
test_indices = np.where(test_mask)[0]

# For each test origin, predict next 14 days
# Origin = last known day; predict origin+1 .. origin+14
baseline_preds = {}  # {origin_idx: (14, 14_segments)}
baseline_actuals = {}

for origin_idx in test_indices:
    if origin_idx + HORIZON > len(ct_arr):
        break
    if origin_idx < LOOKBACK:
        continue
    preds = np.zeros((HORIZON, N_SEGMENTS))
    actuals = ct_arr[origin_idx:origin_idx + HORIZON]  # (14, 14)
    for s in range(N_SEGMENTS):
        history = ct_arr[origin_idx - LOOKBACK:origin_idx, s]
        preds[:, s] = baseline_method(history)
    baseline_preds[origin_idx] = preds
    baseline_actuals[origin_idx] = actuals

print(f'Baseline: {len(baseline_preds)} test forecast origins generated')

## 5. Dataset & DataLoader

In [ ]:
class CTDataset(Dataset):
    def __init__(self, ct_arr, mv_arr, day_type_arr, indices, scalers_ct=None, scalers_mv=None, fit=False):
        """
        ct_arr: (n_days, n_segments)
        mv_arr: (n_days, n_segments)
        day_type_arr: (n_days,)
        indices: list of (origin_idx, segment_idx) tuples
        """
        self.ct = ct_arr
        self.mv = mv_arr
        self.day_type = day_type_arr
        self.indices = indices
        
        # Normalize per segment
        if fit:
            self.scalers_ct = []
            self.scalers_mv = []
            for s in range(ct_arr.shape[1]):
                sc_ct = StandardScaler()
                sc_mv = StandardScaler()
                # Fit on all data available to training origins
                train_origins = [idx for idx, seg in indices]
                max_origin = max(train_origins)
                sc_ct.fit(ct_arr[:max_origin, s].reshape(-1, 1))
                sc_mv.fit(mv_arr[:max_origin, s].reshape(-1, 1))
                self.scalers_ct.append(sc_ct)
                self.scalers_mv.append(sc_mv)
        else:
            self.scalers_ct = scalers_ct
            self.scalers_mv = scalers_mv
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        origin, seg = self.indices[idx]
        
        # Past CT and MV (28 days ending at origin-1, i.e., the day before the forecast starts)
        past_ct = self.ct[origin - LOOKBACK:origin, seg].copy()
        past_mv = self.mv[origin - LOOKBACK:origin, seg].copy()
        
        # Future MV (14 days starting at origin)
        future_mv = self.mv[origin:origin + HORIZON, seg].copy()
        
        # Day type (past 28 + future 14)
        past_day_type = self.day_type[origin - LOOKBACK:origin].astype(np.float32)
        future_day_type = self.day_type[origin:origin + HORIZON].astype(np.float32)
        
        # Target: future CT
        target_ct = self.ct[origin:origin + HORIZON, seg].copy()
        
        # Normalize
        past_ct = self.scalers_ct[seg].transform(past_ct.reshape(-1, 1)).flatten()
        past_mv = self.scalers_mv[seg].transform(past_mv.reshape(-1, 1)).flatten()
        future_mv = self.scalers_mv[seg].transform(future_mv.reshape(-1, 1)).flatten()
        target_ct_norm = self.scalers_ct[seg].transform(target_ct.reshape(-1, 1)).flatten()
        
        # Stack past sequence: (28, 3) -> ct, mv, day_type
        past_seq = np.stack([past_ct, past_mv, past_day_type / 2.0], axis=-1)  # normalize day_type to [0,1]
        
        # Future features: (14, 2) -> mv, day_type
        future_feat = np.stack([future_mv, future_day_type / 2.0], axis=-1)
        
        return {
            'past_seq': torch.tensor(past_seq, dtype=torch.float32),
            'future_feat': torch.tensor(future_feat, dtype=torch.float32),
            'segment_id': torch.tensor(seg, dtype=torch.long),
            'target': torch.tensor(target_ct_norm, dtype=torch.float32),
            'target_raw': torch.tensor(target_ct, dtype=torch.float32),
        }


# Build indices: (origin_idx, segment_idx)
ct_values = ct_gt[SEGMENTS].values
mv_values = mv_pred[SEGMENTS].values
dt_values = day_type_df['day_type'].values

test_start_idx = np.where(ct_gt['hist_date'] >= TEST_START)[0][0]

train_indices = []
test_indices_ds = []

for origin in range(LOOKBACK, len(ct_values) - HORIZON + 1):
    for seg in range(N_SEGMENTS):
        if origin < test_start_idx:
            train_indices.append((origin, seg))
        else:
            test_indices_ds.append((origin, seg))

print(f'Train samples: {len(train_indices)}, Test samples: {len(test_indices_ds)}')

# Create datasets
train_dataset = CTDataset(ct_values, mv_values, dt_values, train_indices, fit=True)
test_dataset = CTDataset(ct_values, mv_values, dt_values, test_indices_ds,
                         scalers_ct=train_dataset.scalers_ct, scalers_mv=train_dataset.scalers_mv)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

## 6. Model Definition

In [ ]:
class DirectCTModel(nn.Module):
    def __init__(self, n_segments=14, past_features=3, future_features=2,
                 hidden_size=64, num_layers=2, embed_dim=8, horizon=14):
        super().__init__()
        self.hidden_size = hidden_size
        self.horizon = horizon
        
        # Segment embedding
        self.seg_embed = nn.Embedding(n_segments, embed_dim)
        
        # Shared LSTM backbone for past sequence (28 timesteps, 3 features)
        self.lstm = nn.LSTM(
            input_size=past_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        # Future MV + day_type encoder
        self.future_encoder = nn.Sequential(
            nn.Linear(horizon * future_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        
        # Combined feature size: hidden_size + 32 (future) + embed_dim
        combined_dim = hidden_size + 32 + embed_dim
        
        # Per-segment heads
        self.segment_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(combined_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, horizon)
            )
            for _ in range(n_segments)
        ])
    
    def forward(self, past_seq, future_feat, segment_id):
        batch_size = past_seq.size(0)
        
        # LSTM on past sequence
        lstm_out, (h_n, _) = self.lstm(past_seq)  # h_n: (num_layers, batch, hidden)
        lstm_feat = h_n[-1]  # last layer hidden state: (batch, hidden)
        
        # Encode future features
        future_flat = future_feat.view(batch_size, -1)
        future_enc = self.future_encoder(future_flat)  # (batch, 32)
        
        # Segment embedding
        seg_emb = self.seg_embed(segment_id)  # (batch, embed_dim)
        
        # Concatenate
        combined = torch.cat([lstm_feat, future_enc, seg_emb], dim=-1)
        
        # Route to per-segment heads
        output = torch.zeros(batch_size, self.horizon, device=past_seq.device)
        for seg_idx in range(len(self.segment_heads)):
            mask = (segment_id == seg_idx)
            if mask.any():
                output[mask] = self.segment_heads[seg_idx](combined[mask])
        
        return output


model = DirectCTModel().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 7. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
N_EPOCHS = 100

train_losses = []
test_losses = []

for epoch in range(N_EPOCHS):
    # Train
    model.train()
    epoch_train_loss = 0.0
    n_train_batches = 0
    for batch in train_loader:
        past_seq = batch['past_seq'].to(device)
        future_feat = batch['future_feat'].to(device)
        seg_id = batch['segment_id'].to(device)
        target = batch['target'].to(device)
        
        optimizer.zero_grad()
        pred = model(past_seq, future_feat, seg_id)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
        n_train_batches += 1
    
    # Test loss
    model.eval()
    epoch_test_loss = 0.0
    n_test_batches = 0
    with torch.no_grad():
        for batch in test_loader:
            past_seq = batch['past_seq'].to(device)
            future_feat = batch['future_feat'].to(device)
            seg_id = batch['segment_id'].to(device)
            target = batch['target'].to(device)
            
            pred = model(past_seq, future_feat, seg_id)
            loss = criterion(pred, target)
            epoch_test_loss += loss.item()
            n_test_batches += 1
    
    avg_train = epoch_train_loss / n_train_batches
    avg_test = epoch_test_loss / n_test_batches
    train_losses.append(avg_train)
    test_losses.append(avg_test)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | Train Loss: {avg_train:.6f} | Test Loss: {avg_test:.6f}')

# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train Loss')
ax.plot(test_losses, label='Test Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (normalized)')
ax.set_title('Training Curve')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Evaluation

In [ ]:
def compute_mape(actual, predicted):
    """MAPE in percentage."""
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


# Generate model predictions on test set
model.eval()
model_preds_raw = {}  # {(origin, seg): pred_14days}

with torch.no_grad():
    for batch in test_loader:
        past_seq = batch['past_seq'].to(device)
        future_feat = batch['future_feat'].to(device)
        seg_id = batch['segment_id'].to(device)
        
        pred_norm = model(past_seq, future_feat, seg_id).cpu().numpy()
        seg_ids = batch['segment_id'].numpy()
        target_raw = batch['target_raw'].numpy()
        
        # Inverse transform predictions
        for i in range(len(pred_norm)):
            s = seg_ids[i]
            pred_original = train_dataset.scalers_ct[s].inverse_transform(
                pred_norm[i].reshape(-1, 1)
            ).flatten()
            # Find the corresponding index in test_indices_ds
            sample_idx = len(model_preds_raw)
            # Store by batch order
            model_preds_raw[len(model_preds_raw)] = {
                'pred': pred_original,
                'actual': target_raw[i],
                'seg': s
            }

# Re-organize: group by origin
# test_indices_ds is [(origin, seg), ...] in order
model_by_origin = {}  # {origin_idx: {seg: pred_14days}}
actual_by_origin = {}  # {origin_idx: {seg: actual_14days}}

for i, (origin, seg) in enumerate(test_indices_ds):
    if origin not in model_by_origin:
        model_by_origin[origin] = {}
        actual_by_origin[origin] = {}
    model_by_origin[origin][seg] = model_preds_raw[i]['pred']
    actual_by_origin[origin][seg] = model_preds_raw[i]['actual']

print(f'Model predictions generated for {len(model_by_origin)} test origins')

In [ ]:
# Compute MAPE per origin for both methods
origins = sorted(set(model_by_origin.keys()) & set(baseline_preds.keys()))
origin_dates = [pd.Timestamp(date_arr[o]) for o in origins]

mape_baseline_per_origin = []
mape_model_per_origin = []
mape_baseline_per_seg = {s: [] for s in range(N_SEGMENTS)}
mape_model_per_seg = {s: [] for s in range(N_SEGMENTS)}

for origin in origins:
    # Baseline MAPE (avg across segments)
    b_mapes = []
    m_mapes = []
    for s in range(N_SEGMENTS):
        actual = actual_by_origin[origin][s]
        b_pred = baseline_preds[origin][:, s]
        m_pred = model_by_origin[origin][s]
        
        bm = compute_mape(actual, b_pred)
        mm = compute_mape(actual, m_pred)
        b_mapes.append(bm)
        m_mapes.append(mm)
        mape_baseline_per_seg[s].append(bm)
        mape_model_per_seg[s].append(mm)
    
    mape_baseline_per_origin.append(np.mean(b_mapes))
    mape_model_per_origin.append(np.mean(m_mapes))

# Overall MAPE
avg_mape_baseline = np.mean(mape_baseline_per_origin)
avg_mape_model = np.mean(mape_model_per_origin)

print('='*50)
print(f'Overall Average MAPE (across all origins & segments)')
print(f'  baseline_method: {avg_mape_baseline:.2f}%')
print(f'  direct_ct:       {avg_mape_model:.2f}%')
print('='*50)

# Per-segment average MAPE
print(f'\nPer-Segment Average MAPE:')
print(f'{"Segment":<10} {"baseline_method":>16} {"direct_ct":>12}')
print('-' * 40)
for s in range(N_SEGMENTS):
    bm = np.mean(mape_baseline_per_seg[s])
    mm = np.mean(mape_model_per_seg[s])
    print(f'{SEGMENTS[s]:<10} {bm:>15.2f}% {mm:>11.2f}%')

## 9. Visualization

In [ ]:
# Plot 1: Running MAPE over test period
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(origin_dates, mape_baseline_per_origin, label='baseline_method', alpha=0.8)
ax.plot(origin_dates, mape_model_per_origin, label='direct_ct', alpha=0.8)
ax.set_xlabel('Forecast Origin Date')
ax.set_ylabel('MAPE (%)')
ax.set_title('Running MAPE Over Test Period (avg across segments)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: CT over test period — actual vs baseline vs direct_ct
# For each segment, plot the 1-day-ahead prediction (first day of each 14-day window)
plot_segments = SEGMENTS[:6]  # show 6 segments
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for i, seg in enumerate(plot_segments):
    s = SEGMENTS.index(seg)
    
    # Collect 1-day-ahead predictions
    dates_plot = []
    actual_vals = []
    baseline_vals = []
    model_vals = []
    
    for origin in origins:
        dates_plot.append(pd.Timestamp(date_arr[origin]))
        actual_vals.append(actual_by_origin[origin][s][0])  # day 1
        baseline_vals.append(baseline_preds[origin][0, s])  # day 1
        model_vals.append(model_by_origin[origin][s][0])    # day 1
    
    axes[i].plot(dates_plot, actual_vals, label='Actual', linewidth=1.5, color='black')
    axes[i].plot(dates_plot, baseline_vals, label='baseline_method', linewidth=1, alpha=0.8, linestyle='--')
    axes[i].plot(dates_plot, model_vals, label='direct_ct', linewidth=1, alpha=0.8, linestyle='-.')
    axes[i].set_title(f'{seg} — 1-day-ahead CT')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted CT Over Test Period (1-day-ahead)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Bar chart — average MAPE per segment
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(N_SEGMENTS)
width = 0.35

baseline_seg_mape = [np.mean(mape_baseline_per_seg[s]) for s in range(N_SEGMENTS)]
model_seg_mape = [np.mean(mape_model_per_seg[s]) for s in range(N_SEGMENTS)]

bars1 = ax.bar(x - width/2, baseline_seg_mape, width, label='baseline_method', color='steelblue')
bars2 = ax.bar(x + width/2, model_seg_mape, width, label='direct_ct', color='coral')

ax.set_xlabel('Segment')
ax.set_ylabel('Average MAPE (%)')
ax.set_title('Average MAPE Per Segment')
ax.set_xticks(x)
ax.set_xticklabels(SEGMENTS)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Full 14-day forecast example for one origin
example_origin = origins[len(origins) // 2]  # mid-test period
example_seg = 0  # G1

forecast_dates = pd.date_range(date_arr[example_origin], periods=HORIZON, freq='D')
actual_14 = actual_by_origin[example_origin][example_seg]
baseline_14 = baseline_preds[example_origin][:, example_seg]
model_14 = model_by_origin[example_origin][example_seg]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(forecast_dates, actual_14, 'ko-', label='Actual', linewidth=2)
ax.plot(forecast_dates, baseline_14, 's--', label='baseline_method', alpha=0.8)
ax.plot(forecast_dates, model_14, 'd-.', label='direct_ct', alpha=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('CT (seconds)')
ax.set_title(f'14-Day Forecast Example — {SEGMENTS[example_seg]} (origin: {pd.Timestamp(date_arr[example_origin]).date()})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nExample MAPE for this window:')
print(f'  baseline_method: {compute_mape(actual_14, baseline_14):.2f}%')
print(f'  direct_ct:       {compute_mape(actual_14, model_14):.2f}%')